# Paper 24 backbone on YOLO26n: LAE-RPD down-sampling + UCDC-Lite context

| Part | Source |
|---|---|
| **Backbone (layers 0–10)** | **Paper 24:** **LAE-RPD** at layers 3, 5, 7 (stride-2 Conv) and/or **UCDC-Lite** at layer 9 (SPPF) |
| Neck FPN + PAN (layers 11–22) | YOLO26n base, unchanged (HPG-SIF and the neck LAE-RPD of Paper 24 are not used) |
| Head (Detect, layer 23) | YOLO26n base, unchanged (one-to-one, NMS-free) |

Three variants are trained:

| Run | Backbone |
|---|---|
| `bb_laerpd` | LAE-RPD instead of the stride-2 Conv at layers 3, 5, 7 |
| `bb_ucdc` | UCDC-Lite instead of the SPPF at layer 9 |
| `bb_laerpd_ucdc` | both together (the Paper 24 backbone) |

The baseline ("Standard YOLO26n Backbone") is **not trained here**; its values are already filled in the final table.

**Kaggle settings:** Accelerator = **GPU T4 x1**, Internet = **On**, keep the notebook **Private**

## 1. Setup

In [ ]:
# ultralytics version is pinned: use the same version on Kaggle, Raspberry Pi 5 and Jetson
!pip install -q ultralytics==8.4.137 roboflow

In [ ]:
import os, sys, gc, shutil, random
from copy import deepcopy
from pathlib import Path

import numpy as np
import pandas as pd
import torch, torchvision
import ultralytics

print("ultralytics", ultralytics.__version__, "| torch", torch.__version__, "| torchvision", torchvision.__version__)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE (turn on the GPU accelerator)")

WORK = Path("/kaggle/working")
os.chdir(WORK)
sys.path.insert(0, str(WORK))  # so fdr_backbone.py can be imported

## 2. Dataset (download from Roboflow)

The Roboflow API key is written in the cell below. Keep this notebook **Private** on Kaggle.

In [ ]:
import os, yaml
from roboflow import Roboflow

os.chdir("/kaggle/working")

# Roboflow API key (keep this notebook Private on Kaggle)
API_KEY = "CxHgc1Nn3FK5vYNsBtJW"

rf = Roboflow(api_key=API_KEY)
project = rf.workspace("md-mahibullah").project("digitalaccess-2-copy")
dataset = project.version(4).download("yolo26")  # Roboflow YOLO26 export (same label format as YOLOv8)

# Rename and fix paths
new = "/kaggle/working/digitalaccess"
if os.path.exists(dataset.location) and not os.path.exists(new):
    os.rename(dataset.location, new)

# Update YAML
yaml_path = f"{new}/data.yaml"
with open(yaml_path) as f:
    data = yaml.safe_load(f)
data["path"], data["train"], data["val"] = new, "train/images", "valid/images"
if os.path.exists(f"{new}/test/images"):
    data["test"] = "test/images"
else:
    data.pop("test", None)
with open(yaml_path, "w") as f:
    yaml.dump(data, f)

# the following cells use these variables
DATA_ROOT = Path(new)
DATA_YAML = yaml_path
names = data["names"]
assert data.get("test"), "No test split: the evaluation and FPS steps need a test split"

for s in ["train", "valid", "test"]:
    print(f"{s:6s}: {len(list((DATA_ROOT / s / 'images').glob('*')))} images")
print("nc =", len(names))
print("✅ Dataset prepared!")

## 3. Paper 24 backbone: LAE-RPD and UCDC-Lite

**LAE-RPD** (Paper 24, Eq. 1–6), content-adaptive down-sampling:
- *attention branch*: AvgPool 3×3 (stride 1) → 1×1 conv; each 2×2 neighbourhood gives 4 candidate positions, and a
  softmax across the 4 positions gives their weights;
- *candidate branch*: grouped 3×3 stride-2 conv (G = 16 channels per group) gives 4 candidate responses per output position;
- output = weighted sum of the 4 candidates, then BN + SiLU.

**UCDC-Lite** (Paper 24, Eq. 7–14): 1×1 conv to half the channels (X1) → 3×3 dilated conv d = 2 (D1) → 3×3 d = 4 (D2)
→ 3×3 d = 2 on concat(D1, D2) (D3) → 1×1 conv on concat(X1, D3). The resolution stays the same.

**Where they sit in YOLO26n.**

| Layer | Base YOLO26n | `bb_laerpd` | `bb_ucdc` | `bb_laerpd_ucdc` | Output |
|---|---|---|---|---|---|
| 3 | Conv 3×3 s2 (64 → 64) | **LAE-RPD** | Conv | **LAE-RPD** | P3/8 (80 × 80) |
| 5 | Conv 3×3 s2 (128 → 128) | **LAE-RPD** | Conv | **LAE-RPD** | P4/16 (40 × 40) |
| 7 | Conv 3×3 s2 (128 → 256) | **LAE-RPD** | Conv | **LAE-RPD** | P5/32 (20 × 20) |
| 9 | SPPF | SPPF | **UCDC-Lite** | **UCDC-Lite** | P5/32 (20 × 20) |

Input and output channels stay the same, so no layer index changes and the neck connections stay the same.

**Pretrained weights.** `yolo26n.pt` is loaded first, then the layers are changed. LAE-RPD layers start from scratch;
the first 1×1 conv of UCDC-Lite (256 → 128) takes the weights of the SPPF `cv1` (same shape); the rest of UCDC-Lite is new.

**NCNN note.** Paper 24 writes the softmax on a 5-D tensor (C × H × W × 4). Here it is written with 4-D operations
(`maximum`, `exp`, add, divide); the result is mathematically identical.

**Assumptions** (not stated in Paper 24): how LAE-RPD changes the channel count (layer 7: 128 → 256; here the candidate
conv outputs 4 × c2 channels and the attention conv outputs c2), BN + SiLU after LAE-RPD, and no SPPF shortcut in UCDC-Lite.

In [ ]:
%%writefile fdr_backbone.py
"""Paper 24 backbone for YOLO26n: LAE-RPD at layers 3, 5, 7 (stride-2 Conv) and/or UCDC-Lite at layer 9 (SPPF)."""
import torch
import torch.nn as nn

from ultralytics.models.yolo.detect import DetectionTrainer
from ultralytics.nn.modules.block import SPPF
from ultralytics.nn.modules.conv import Conv
from ultralytics.nn.tasks import DetectionModel
from ultralytics.utils import RANK

LAERPD_LAYERS = (3, 5, 7)
UCDC_LAYER = 9
SUBS = ((0, 0), (1, 0), (0, 1), (1, 1))  # the 4 positions of each 2x2 neighbourhood


class LAERPD(nn.Module):
    """LAE-based response-preserving down-sampling: softmax weights over the 4 positions of each 2x2 block x 4 grouped-conv candidates."""

    def __init__(self, c1, c2, G=16):
        super().__init__()
        g = max(1, c1 // G)  # number of conv groups (G channels per group)
        assert c1 % g == 0 and (4 * c2) % g == 0, f"group count {g} does not divide {c1} / {4 * c2}"
        self.pool = nn.AvgPool2d(3, 1, 1)
        self.att = nn.Conv2d(c1, c2, 1)
        self.cand = nn.Conv2d(c1, 4 * c2, 3, 2, 1, groups=g, bias=False)
        self.bn = nn.BatchNorm2d(c2)
        self.act = nn.SiLU()

    def weights(self, x):
        """Softmax over the 4 candidate positions, written with 4-D ops only (NCNN friendly). Returns 4 maps that sum to 1."""
        a = self.att(self.pool(x))
        logits = [a[..., i::2, j::2] for i, j in SUBS]
        mx = torch.maximum(torch.maximum(logits[0], logits[1]), torch.maximum(logits[2], logits[3]))
        e = [torch.exp(l - mx) for l in logits]
        s = e[0] + e[1] + e[2] + e[3]
        return [w / s for w in e]

    def forward(self, x):
        w = self.weights(x)
        v = self.cand(x).chunk(4, 1)
        return self.act(self.bn(w[0] * v[0] + w[1] * v[1] + w[2] * v[2] + w[3] * v[3]))


class UCDCLite(nn.Module):
    """Lightweight U-shaped dilated context aggregation: 1x1 -> 3x3 d2 -> 3x3 d4 -> 3x3 d2 on concat -> 1x1 on concat with X1."""

    def __init__(self, c1, c2):
        super().__init__()
        cm = c2 // 2
        self.cv1 = Conv(c1, cm, 1, 1)
        self.d1 = Conv(cm, cm // 2, 3, 1, d=2)
        self.d2 = Conv(cm // 2, cm // 2, 3, 1, d=4)
        self.d3 = Conv(cm, cm, 3, 1, d=2)
        self.cv2 = Conv(2 * cm, c2, 1, 1)

    def forward(self, x):
        x1 = self.cv1(x)
        d1 = self.d1(x1)
        d2 = self.d2(d1)
        d3 = self.d3(torch.cat((d1, d2), 1))
        return self.cv2(torch.cat((x1, d3), 1))


def _swap(model, i, new, name):
    old = model.model[i]
    new = new.to(next(old.parameters()).device)
    new.i, new.f, new.type = old.i, old.f, name  # keep the routing attributes used by the model's forward
    new.np = sum(p.numel() for p in new.parameters())
    model.model[i] = new


def add_laerpd(model, layers=LAERPD_LAYERS):
    """Replace the 3x3 stride-2 Conv at the given backbone layers with LAE-RPD. Returns the model."""
    for i in layers:
        old = model.model[i]
        assert isinstance(old, Conv) and old.conv.kernel_size == (3, 3) and old.conv.stride == (2, 2), f"layer {i} is not a 3x3 s2 Conv"
        _swap(model, i, LAERPD(old.conv.in_channels, old.conv.out_channels), "LAERPD")
    return model


def add_ucdc(model, layer=UCDC_LAYER):
    """Replace the SPPF at the given layer with UCDC-Lite; its first 1x1 conv takes the SPPF cv1 weights. Returns the model."""
    old = model.model[layer]
    assert isinstance(old, SPPF), f"layer {layer} is {type(old).__name__}, not SPPF"
    new = UCDCLite(old.cv1.conv.in_channels, old.cv2.conv.out_channels)
    if new.cv1.conv.weight.shape == old.cv1.conv.weight.shape:
        new.cv1.load_state_dict(old.cv1.state_dict())
    _swap(model, layer, new, "UCDCLite")
    return model


class FDRTrainer(DetectionTrainer):
    """Loads the pretrained weights first, then applies LAE-RPD and/or UCDC-Lite."""

    LAERPD = False
    UCDC = False

    def get_model(self, cfg=None, weights=None, verbose=True):
        model = DetectionModel(cfg, nc=self.data["nc"], ch=self.data["channels"], verbose=False)
        if weights:
            model.load(weights)
        parts = []
        if self.LAERPD:
            add_laerpd(model)
            parts.append(f"LAE-RPD at layers {LAERPD_LAYERS}")
        if self.UCDC:
            add_ucdc(model)
            parts.append(f"UCDC-Lite at layer {UCDC_LAYER}")
        if RANK in {-1, 0}:
            print("Paper 24 backbone: " + ", ".join(parts))
        if verbose and RANK in {-1, 0}:
            model.info()
        return model


class LAERPDTrainer(FDRTrainer):
    LAERPD = True


class UCDCTrainer(FDRTrainer):
    UCDC = True


class LAERPDUCDCTrainer(FDRTrainer):
    LAERPD = True
    UCDC = True


In [ ]:
import importlib, fdr_backbone
importlib.reload(fdr_backbone)
from fdr_backbone import add_laerpd, add_ucdc, LAERPDTrainer, UCDCTrainer, LAERPDUCDCTrainer
from ultralytics.nn.tasks import DetectionModel, yaml_model_load
from ultralytics.utils.torch_utils import get_flops

def fused_params_gflops(model, imgsz=640):
    m = deepcopy(model).float().cpu().eval()
    m.fuse(verbose=False)
    return sum(p.numel() for p in m.parameters()) / 1e6, get_flops(m, imgsz)

# structure check (untrained): base vs the three Paper 24 backbones
base = DetectionModel(yaml_model_load("yolo26n.yaml"), nc=len(names), verbose=False).eval()
m_lae = add_laerpd(deepcopy(base)).eval()
m_ucdc = add_ucdc(deepcopy(base)).eval()
m_both = add_ucdc(add_laerpd(deepcopy(base))).eval()
with torch.no_grad():
    for m in (m_lae, m_ucdc, m_both):
        _ = m(torch.zeros(1, 3, 640, 640))  # forward check
    x, shapes = torch.zeros(1, 3, 640, 640), {}
    for i, m in enumerate(m_both.model[:10]):  # backbone layers 0-9 are sequential (from = -1)
        if i == 3:
            w = m.weights(x)
            print("LAE-RPD weights sum to 1:", torch.allclose(w[0] + w[1] + w[2] + w[3], torch.ones_like(w[0]), atol=1e-5))
        x = m(x)
        if i in (3, 5, 7, 9):
            shapes[i] = tuple(x.shape[1:])
print("layers 3, 5, 7, 9 (both):", [m_both.model[i].type for i in (3, 5, 7, 9)], "| output (C, H, W):", shapes)
for tag, m in (("base", base), ("LAE-RPD", m_lae), ("UCDC-Lite", m_ucdc), ("LAE-RPD + UCDC-Lite", m_both)):
    pm, gf = fused_params_gflops(m)
    print(f"{tag:20s} Params {pm:.4f} M | GFLOPs {gf:.3f}")

## 4. Training

Three Paper 24 backbone variants are trained one after another with identical settings. To skip a variant, put `#` in front of its line in `VARIANTS`.

In [ ]:
from ultralytics import YOLO

EPOCHS  = 100
IMGSZ   = 640
BATCH   = 16
WORKERS = 2
SEED    = 0
PROJECT = str(WORK / "runs")

COMMON = dict(
    data=DATA_YAML, epochs=EPOCHS, imgsz=IMGSZ, batch=BATCH, seed=SEED, deterministic=True,
    pretrained="yolo26n.pt", device=0, workers=WORKERS, cache=False, patience=30, cos_lr=False,
    project=PROJECT, exist_ok=True, plots=True,
    # channels_last=False: ultralytics' MuSGD crashes on channels_last weights ("view size is not compatible").
    optimizer="MuSGD", lr0=0.01, momentum=0.9, channels_last=False,
)

# (row label in the table, run name, trainer)
VARIANTS = [
    ("LAE-RPD Backbone (Paper 24)",             "bb_laerpd",      LAERPDTrainer),
    ("UCDC-Lite Backbone (Paper 24)",           "bb_ucdc",        UCDCTrainer),
    ("LAE-RPD + UCDC-Lite Backbone (Paper 24)", "bb_laerpd_ucdc", LAERPDUCDCTrainer),
]

def best_path(run):
    return Path(PROJECT) / f"{run}_s{SEED}" / "weights" / "best.pt"

for label, run, trainer in VARIANTS:
    print(f"\n========== {label} ({run}) ==========")
    model = YOLO("yolo26n.yaml")
    model.train(trainer=trainer, **dict(COMMON, name=f"{run}_s{SEED}"))
    del model; gc.collect(); torch.cuda.empty_cache()

## 5. Test-set evaluation: Precision, Recall, F1-score, mAP@50, Params, GFLOPs

Params and GFLOPs are counted on the fused inference model, the same way as the baseline row.

In [ ]:
import fdr_backbone  # must be imported before loading a Paper 24 checkpoint

rows = []
for label, run, _ in VARIANTS:
    w = best_path(run)
    if not w.exists():
        print(f"{run}: best.pt not found, train it first"); continue
    r = YOLO(str(w)).val(data=DATA_YAML, split="test", imgsz=IMGSZ, batch=16, device=0, plots=False, verbose=False)
    P, R = float(r.box.mp), float(r.box.mr)
    params, gflops = fused_params_gflops(YOLO(str(w)).model, IMGSZ)
    rows.append({"run": run, "Backbone Variant": label,
                 "Params (M)": round(params, 2), "GFLOPs": round(gflops, 2),
                 "Precision": round(P, 3), "Recall": round(R, 3),
                 "F1-score": round(2 * P * R / (P + R + 1e-9), 3),
                 "mAP@50": round(float(r.box.map50), 3)})
METRICS = pd.DataFrame(rows)
METRICS.drop(columns="run")

## 6. Package for Raspberry Pi 5 and Jetson

Creates `paper24_backbone_edge_bench.zip` in the **Output** tab with:

| Item | Used on |
|---|---|
| `bb_laerpd_ncnn_model/`, `bb_ucdc_ncnn_model/`, `bb_laerpd_ucdc_ncnn_model/` (NCNN, imgsz 640) | Raspberry Pi 5 |
| `bb_laerpd.pt`, `bb_ucdc.pt`, `bb_laerpd_ucdc.pt` | Jetson |
| `bench_images/` (50 real test images, seed 0) | both |
| `fdr_backbone.py` | Jetson (needed to load the Paper 24 `.pt`) |
| `pi5_bench.py`, `jetson_bench.py` | benchmark scripts |

Both scripts: batch 1, imgsz 640, cycle through the 50 real test images (loaded into memory first),
100 warm-up runs, then 1000 timed runs end-to-end (preprocessing + inference + postprocessing), and print the **median** ms and FPS.
Unzip into its own folder, e.g. `python -m zipfile -e paper24_backbone_edge_bench.zip paper24_backbone_edge_bench`.

In [ ]:
EDGE_IMGSZ = 640
N_BENCH_IMAGES = 50
EDGE = WORK / "paper24_backbone_edge_bench"
shutil.rmtree(EDGE, ignore_errors=True)
EDGE.mkdir()
shutil.copy("fdr_backbone.py", EDGE / "fdr_backbone.py")

# 50 real test images (fixed seed, so every model and device uses the same images)
test_imgs = sorted(p for p in (DATA_ROOT / "test" / "images").glob("*") if p.suffix.lower() in {".jpg", ".jpeg", ".png", ".bmp"})
random.Random(SEED).shuffle(test_imgs)
(EDGE / "bench_images").mkdir()
for p in test_imgs[:N_BENCH_IMAGES]:
    shutil.copy(p, EDGE / "bench_images" / p.name)
print(f"bench_images: {len(list((EDGE / 'bench_images').glob('*')))} real test images")

for _, run, _ in VARIANTS:
    w = best_path(run)
    if not w.exists():
        print(f"{run}: best.pt not found, skipped"); continue
    shutil.copy(w, EDGE / f"{run}.pt")
    try:
        p = YOLO(str(w)).export(format="ncnn", imgsz=EDGE_IMGSZ, half=False)
        shutil.copytree(p, EDGE / f"{run}_ncnn_model")  # ultralytics needs the "_ncnn_model" suffix
        print(f"✅ {run:16s} -> {run}.pt, {run}_ncnn_model/")
    except Exception as e:
        print(f"❌ {run:16s} NCNN export failed: {e}  (the .pt for Jetson is still packaged)")

COMMON_BENCH = """
import glob, statistics, sys, time
import cv2

IMGSZ, WARMUP, RUNS = {imgsz}, 100, 1000
imgs = [cv2.imread(p) for p in sorted(glob.glob("bench_images/*"))]
imgs = [im for im in imgs if im is not None]
if not imgs:
    sys.exit("No images found in bench_images/. Run this script inside the unzipped folder.")
print(f"{{len(imgs)}} real test images, imgsz {{IMGSZ}}, batch 1, {{WARMUP}} warm-up + {{RUNS}} timed runs (median)")


def bench(model, **kw):
    for i in range(WARMUP):
        model.predict(imgs[i % len(imgs)], imgsz=IMGSZ, verbose=False, **kw)
    times = []
    for i in range(RUNS):
        t = time.perf_counter()
        model.predict(imgs[i % len(imgs)], imgsz=IMGSZ, verbose=False, **kw)
        times.append((time.perf_counter() - t) * 1000)
    return statistics.median(times)
""".format(imgsz=EDGE_IMGSZ)

PI5 = '''# Raspberry Pi 5 benchmark (NCNN). Run inside the unzipped folder:  python pi5_bench.py
from ultralytics import YOLO
''' + COMMON_BENCH + '''
models = sorted(glob.glob("*_ncnn_model"))
if not models:
    sys.exit("No *_ncnn_model folders found.")
for path in models:
    ms = bench(YOLO(path, task="detect"))
    print(f"{path.replace('_ncnn_model', ''):12s}  median {ms:8.2f} ms   {1000 / ms:7.2f} FPS")
'''

JETSON = '''# Jetson benchmark (PyTorch, CUDA FP16). Run inside the unzipped folder:  python jetson_bench.py
import sys
import torch

if not torch.cuda.is_available():
    sys.exit("WARNING: CUDA GPU not available. Install the NVIDIA Jetson PyTorch wheel for your JetPack, then run again.")
print("GPU:", torch.cuda.get_device_name(0))

sys.path.insert(0, ".")
import fdr_backbone  # needed to load the Paper 24 checkpoints
from ultralytics import YOLO
''' + COMMON_BENCH + '''
models = sorted(glob.glob("*.pt"))
if not models:
    sys.exit("No .pt files found.")
for path in models:
    ms = bench(YOLO(path, task="detect"), device=0, half=True)
    print(f"{path.replace('.pt', ''):12s}  median {ms:8.2f} ms   {1000 / ms:7.2f} FPS")
'''

(EDGE / "pi5_bench.py").write_text(PI5)
(EDGE / "jetson_bench.py").write_text(JETSON)
shutil.make_archive(str(WORK / "paper24_backbone_edge_bench"), "zip", EDGE)
print("\nDownload paper24_backbone_edge_bench.zip from the Output tab.")

### How to run on the devices

**Raspberry Pi 5** (same Pi and same conditions as the baseline measurement)
1. Activate the Python environment that has `ultralytics==8.4.137` and `ncnn`.
2. `echo performance | sudo tee /sys/devices/system/cpu/cpu*/scaling_governor`
3. Go into the unzipped folder and run `python pi5_bench.py`.
4. Run it 3 times with a 2–3 minute pause; use the middle value. Check `vcgencmd measure_temp` and `vcgencmd get_throttled` (`throttled=0x0` = OK).

**Jetson**
1. NVIDIA Jetson PyTorch + torchvision for your JetPack, then `ultralytics==8.4.137`.
2. `sudo nvpmodel -m 0` (check with `sudo nvpmodel -q`) and `sudo jetson_clocks`.
3. Go into the unzipped folder and run `python jetson_bench.py`. Run 3 times; use the middle value.

## 7. Final backbone table

The baseline row is already filled in. Type the median FPS of the three Paper 24 backbone models
(and the baseline Jetson FPS when you have it), keep `None` for a missing value (shown as `--`), then run the cell.
Run section 5 first in the same session (it fills `METRICS`).

In [ ]:
# baseline values (identical YOLO26n model)
BASE_ROW = {"Backbone Variant": "Standard YOLO26n Backbone", "Params (M)": 2.38, "GFLOPs": 5.34,
            "Precision": 0.971, "Recall": 0.937, "F1-score": 0.953, "mAP@50": 0.958,
            "FPS (Rasp-5)": 13.38, "FPS (Jetson)": None}

FPS_PI5    = {"bb_laerpd": None, "bb_ucdc": None, "bb_laerpd_ucdc": None}
FPS_JETSON = {"bb_laerpd": None, "bb_ucdc": None, "bb_laerpd_ucdc": None}

table = METRICS.copy()
table["FPS (Rasp-5)"] = table["run"].map(lambda r: FPS_PI5.get(r))
table["FPS (Jetson)"] = table["run"].map(lambda r: FPS_JETSON.get(r))
COLS = ["Backbone Variant", "Params (M)", "GFLOPs", "Precision", "Recall", "F1-score", "mAP@50", "FPS (Rasp-5)", "FPS (Jetson)"]
table = pd.concat([pd.DataFrame([BASE_ROW]), table[COLS]], ignore_index=True)[COLS]
table = table.astype(object).where(table.notna(), "--")
table.style.hide(axis="index")

## 8. Zip all results

Run this last. It creates **`paper24_backbone_all_results.zip`** in the **Output** tab with:

| Folder in the zip | Contents |
|---|---|
| `paper24_backbone_edge_bench/` | NCNN + `.pt` models, 50 bench images, `fdr_backbone.py`, `pi5_bench.py`, `jetson_bench.py` |
| `weights/` | `bb_laerpd_best.pt`, `bb_ucdc_best.pt`, `bb_laerpd_ucdc_best.pt` |
| `runs/` | full training folder: `best.pt`, `last.pt`, `results.csv`, training curves, confusion matrix, PR curves |

In [ ]:
import zipfile

ALL_ZIP = WORK / "paper24_backbone_all_results.zip"
folders = [WORK / "paper24_backbone_edge_bench", WORK / "runs"]

with zipfile.ZipFile(ALL_ZIP, "w", zipfile.ZIP_DEFLATED) as z:
    for folder in folders:
        if not folder.exists():
            print(f"{folder.name}: not found, skipped"); continue
        n = 0
        for f in folder.rglob("*"):
            if f.is_file():
                z.write(f, f.relative_to(WORK))
                n += 1
        print(f"{folder.name + '/':30s} {n} files")
    for _, run, _ in VARIANTS:
        w = best_path(run)
        if w.exists():
            z.write(w, f"weights/{run}_best.pt")
            print(f"weights/{run}_best.pt")

print(f"\nDownload {ALL_ZIP.name} ({ALL_ZIP.stat().st_size / 1e6:.1f} MB) from the Output tab.")